In [ ]:
import yt_dlp
from yt_dlp.utils import DownloadError

def download_1080p_video(url: str) -> None:
    """
    下載 YouTube 影片（最高 1080p，最終輸出 MP4）。

    需求重點：
    - 預設選擇最佳 <=1080p（可自動回退到 720p/480p...）
    - 強制輸出容器為 MP4
    - 先將縮圖轉成 JPG，再嵌入 MP4
    - 同一輪後處理同時做 Metadata + 縮圖嵌入，降低檔案鎖定衝突
    - Windows 安全檔名（restrictfilenames=True）

    環境假設：
    - ffmpeg 已在 PATH
    - 建議安裝 mutagen（metadata 寫入更穩定）
    - 建議安裝 Node.js（處理 YouTube 新版串流實驗）
    """

    def progress_hook(d):
        status = d.get("status")
        if status == "downloading":
            if not progress_hook._analyzing_printed:
                print("Analyzing...")
                progress_hook._analyzing_printed = True
        elif status == "finished":
            print("Merging...")

    progress_hook._analyzing_printed = False

    ydl_opts = {
        "format": "bestvideo[height<=1080][vcodec^=av01]+bestaudio[ext=m4a]/best[ext=mp4]/best",
        "merge_output_format": "mp4",
        "outtmpl": "%(title)s.%(ext)s",
        "restrictfilenames": True,
        "writethumbnail": True,
        "addmetadata": True,
        "postprocessors": [
            {
                "key": "FFmpegVideoConvertor",
                "preferedformat": "mp4",
            },
            {
                "key": "FFmpegMetadata",
                "add_metadata": True,
                "add_chapters": True,
            },
            {
                "key": "FFmpegThumbnailsConvertor",
                "format": "jpg",
            },
            {
                "key": "EmbedThumbnail",
                "already_have_thumbnail": False,
            },
        ],
        "progress_hooks": [progress_hook],
        "noplaylist": True,
        "quiet": False,
        "no_warnings": False,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        print("Done")
    except DownloadError as e:
        print(f"Download failed: {e}")

if __name__ == "__main__":
    url = input("Enter YouTube URL: ").strip()
    download_1080p_video(url)